# 02 — Auto-Labeling

Two stages:
1. **NSFW oracle** (Falconsai, runs in Colab on T4) — flags obvious nudity
2. **VLM labeling** (NIM primary, Gemini secondary for QA + refusal fallback) — fine-grained tzniut attributes

Resumable. Re-run any cell after timeout.

In [ ]:
import os, pathlib
if not pathlib.Path('zahava-tzniut').exists():
    !git clone https://github.com/YOUR_USERNAME/zahava-tzniut.git
os.chdir('zahava-tzniut')
!pip install -q -r requirements.txt transformers torch
assert pathlib.Path('.env').exists(), 'upload .env first'

In [ ]:
# Pull the latest manifest from HF (written by notebook 01)
from huggingface_hub import hf_hub_download
from pipelines.common import require_env
manifest_path = hf_hub_download(
    repo_id=require_env('HF_DATASET_REPO'),
    filename='collection_deduped.parquet',
    repo_type='dataset',
    local_dir='manifests',
)
print('manifest:', manifest_path)

In [ ]:
# Stage A: NSFW oracle
from pipelines.labeling import nsfw_oracle
nsfw_oracle.run()

In [ ]:
# Stage B: VLM labeling (NIM + Gemini QA at 5%)
from pipelines.labeling import vlm_labeler
vlm_labeler.run(round_name='vlm_round_1')

In [ ]:
# Merge labels and push to HF
from pipelines.labeling.run import run as run_all
run_all(round_name='vlm_round_1', push_to_hf=True)

In [ ]:
# Quick stats on the labeled set
import pandas as pd, json
df = pd.read_parquet('manifests/labels.parquet')
df_nim = df[df['labeler'] == 'nim']
print(f"total labeled (NIM): {len(df_nim)}")
print(f"block rate: {df_nim['block'].mean():.3f}")
print(f"flagged for review: {df['flagged_for_review'].sum()}")
vio = df_nim['violations_json'].apply(json.loads).explode()
print('\ntop violations:')
print(vio.value_counts().head(15))